In [ ]:
# | default_exp transforms/monai/intensity

# Imports

In [ ]:
# | export

from collections.abc import Hashable, Mapping, Sequence
from warnings import warn

import numpy as np
import torch
from einops import rearrange
from monai.config import KeysCollection, NdarrayOrTensor
from monai.config.type_definitions import DtypeLike
from monai.data.meta_obj import get_track_meta
from monai.transforms import MapTransform, Transform
from monai.transforms.utils_pytorch_numpy_unification import clip
from monai.utils.enums import TransformBackends
from monai.utils.type_conversion import convert_data_type, convert_to_tensor

In [ ]:
# | export


class MultiScaleIntensityRange(Transform):
    """Apply multiple intensity windows to a 4-D ``(C, Z Y X)`` tensor and
    concatenate the results along the channel dimension.

    Each window is applied to the **full** ``(C, Z Y X)`` input, producing
    one ``(C, Z Y X)`` output per window.  The ``W_i`` outputs are
    concatenated channel-first to give ``(W_i * C, Z Y X)``.

    Each window is specified **either** by ``(a_min, a_max)`` range **or** by
    ``(window_center, window_width)`` — but not both at the same time for a
    single window.  You may freely mix styles across windows.  Per-window
    ``b_min`` / ``b_max`` control the target range (default ``0 → 1``).

    Args:
        windows: sequence of window specs.  Each element is a dict with
            **exactly one** of the following key groups:

            * ``{"a_min": …, "a_max": …}`` — explicit source range, **or**
            * ``{"window_center": …, "window_width": …}`` — centre/width
              form (converted to ``a_min = center - width/2``,
              ``a_max = center + width/2``).

            Each dict may optionally include:

            * ``"b_min"`` (default ``0.0``) — target range minimum.
            * ``"b_max"`` (default ``1.0``) — target range maximum.

        clip: whether to clip output values to ``[b_min, b_max]`` for every
            window.
        dtype: output dtype; ``None`` keeps the input dtype.
        mode: ``"concat"`` or ``"interleave"``.

            * ``"concat"`` (default): windows concatenated in order —
              ``[w0_ch0, w0_ch1, …, w1_ch0, w1_ch1, …]``
              i.e. ``(W_i * C, Z Y X)``.
            * ``"interleave"``: channels grouped per input channel —
              ``[ch0_w0, ch0_w1, …, ch1_w0, ch1_w1, …]``
              i.e. ``(C * W_i, Z Y X)``.
    """

    MODES = ("concat", "interleave")
    backend = [TransformBackends.TORCH, TransformBackends.NUMPY]

    def __init__(
        self,
        windows: Sequence[dict],
        clip: bool = True,
        dtype: DtypeLike = np.float32,
        mode: str = "concat",
    ) -> None:
        super().__init__()
        assert len(windows) > 0, "Must provide at least one window"
        assert mode in self.MODES, f"mode must be one of {self.MODES}, got '{mode}'"
        self.clip = clip
        self.dtype = dtype
        self.mode = mode
        self._windows = self._parse_windows(windows)

    # ------------------------------------------------------------------
    @staticmethod
    def _parse_windows(
        windows: Sequence[dict],
    ) -> list[dict]:
        """Normalise every window spec into ``{a_min, a_max, b_min, b_max}``."""
        parsed: list[dict] = []
        for i, w in enumerate(windows):
            has_range = "a_min" in w and "a_max" in w
            has_center = "window_center" in w and "window_width" in w
            assert has_range ^ has_center, (
                f"Window {i}: provide exactly one of "
                f"(a_min, a_max) or (window_center, window_width), "
                f"got keys {set(w.keys())}"
            )
            if has_center:
                half = w["window_width"] / 2.0
                a_min = w["window_center"] - half
                a_max = w["window_center"] + half
            else:
                a_min = w["a_min"]
                a_max = w["a_max"]

            parsed.append(
                {
                    "a_min": float(a_min),
                    "a_max": float(a_max),
                    "b_min": float(w.get("b_min", 0.0)),
                    "b_max": float(w.get("b_max", 1.0)),
                }
            )
        return parsed

    def _apply_single_window(
        self, img: NdarrayOrTensor, a_min: float, a_max: float, b_min: float, b_max: float
    ) -> NdarrayOrTensor:
        """Scale ``img`` from ``[a_min, a_max]`` → ``[b_min, b_max]``."""
        if a_max - a_min == 0.0:
            warn("Divide by zero (a_min == a_max)", Warning)
            return img - a_min + b_min

        out = (img - a_min) / (a_max - a_min)
        if (b_min is not None) and (b_max is not None):
            out = out * (b_max - b_min) + b_min
        if self.clip:
            out = clip(out, b_min, b_max)
        return out

    def __call__(self, img: NdarrayOrTensor) -> NdarrayOrTensor:
        """Apply all windows to ``img`` (must be 4-D ``(C, Z Y X)``).

        Output shape is ``(W_i * C, Z Y X)`` for ``"concat"`` mode or
        ``(C * W_i, Z Y X)`` for ``"interleave"`` mode.
        """
        img = convert_to_tensor(img, track_meta=get_track_meta())
        assert img.ndim == 4, (
            f"MultiScaleIntensityRange expects 4-D (C, Z Y X) input, " f"got {img.ndim}-D with shape {tuple(img.shape)}"
        )

        W = len(self._windows)
        C = img.shape[0]

        # Apply each window to the full (C, Z Y X) tensor → list of (C, Z Y X)
        windowed = [self._apply_single_window(img, **w) for w in self._windows]

        # Concat: (W*C, Z Y X) — [w0_ch0, w0_ch1, …, w1_ch0, w1_ch1, …]
        out = torch.cat(windowed, dim=0)
        out = convert_data_type(out, dtype=self.dtype)[0]

        if self.mode == "interleave":
            out = rearrange(out, "(W C) Z Y X -> (C W) Z Y X", C=C, W=W)

        return out


class MultiScaleIntensityRanged(MapTransform):
    """Dictionary-based wrapper of :class:`MultiScaleIntensityRange`.

    Applies multiple intensity windows to each image key and concatenates
    the results along the channel dimension.

    The input tensor for each key must be 4-D ``(C, Z Y X)``.  After
    windowing the shape becomes ``(W_i * C, Z Y X)`` (or
    ``(C * W_i, Z Y X)`` for interleave) with channel ordering controlled
    by ``mode``.

    Args:
        keys: keys of the image tensors to transform.
        windows: sequence of window spec dicts — see
            :class:`MultiScaleIntensityRange` for the format.
        clip: whether to clip to ``[b_min, b_max]``.
        dtype: output dtype; ``None`` keeps input dtype.
        mode: ``"concat"`` or ``"interleave"``.
        allow_missing_keys: if ``True``, do not raise if a key is absent.
    """

    backend = MultiScaleIntensityRange.backend

    def __init__(
        self,
        keys: KeysCollection,
        windows: Sequence[dict],
        clip: bool = True,
        dtype: DtypeLike = np.float32,
        mode: str = "concat",
        allow_missing_keys: bool = False,
    ) -> None:
        super().__init__(keys, allow_missing_keys=allow_missing_keys)
        self.scaler = MultiScaleIntensityRange(windows=windows, clip=clip, dtype=dtype, mode=mode)

    def __call__(self, data: Mapping[Hashable, NdarrayOrTensor]) -> dict[Hashable, NdarrayOrTensor]:
        d = dict(data)
        for key in self.key_iterator(d):
            d[key] = self.scaler(d[key])
        return d

In [ ]:
# --- Test MultiScaleIntensityRange (base transform) ---


import torch

windows_range = [
    {"a_min": -1350.0, "a_max": 150.0},  # lung
    {"a_min": -150.0, "a_max": 250.0},  # mediastinum
    {"a_min": -500.0, "a_max": 1300.0},  # bone
]

# 1) Single-channel 4-D input, concat mode → (W*1, Z Y X) = (3, 8, 8, 8)
img_1ch = torch.linspace(-1024, 3071, 8 * 8 * 8).reshape(1, 8, 8, 8)
out = MultiScaleIntensityRange(windows=windows_range)(img_1ch)
assert out.shape == (3, 8, 8, 8), f"Expected (3,8,8,8), got {out.shape}"
assert out.min() >= 0.0 and out.max() <= 1.0
print(f"[PASS] 1-ch concat: shape={tuple(out.shape)}, range=[{out.min():.3f}, {out.max():.3f}]")

# 2) window_center / window_width form produces same result as equivalent a_min/a_max
windows_center = [
    {"window_center": -600.0, "window_width": 1500.0},
    {"window_center": 50.0, "window_width": 400.0},
    {"window_center": 400.0, "window_width": 1800.0},
]
out_center = MultiScaleIntensityRange(windows=windows_center)(img_1ch)
assert torch.allclose(out, out_center, atol=1e-6), "center/width form should match a_min/a_max form"
print("[PASS] window_center/window_width matches a_min/a_max")

# 3) Mixed styles in a single call
windows_mixed = [
    {"a_min": -1350.0, "a_max": 150.0},
    {"window_center": 50.0, "window_width": 400.0},
]
out_mixed = MultiScaleIntensityRange(windows=windows_mixed)(img_1ch)
assert out_mixed.shape == (2, 8, 8, 8)
print("[PASS] Mixed window styles accepted")

# 4) Custom b_min / b_max per window
windows_custom_b = [
    {"a_min": 0.0, "a_max": 100.0, "b_min": -1.0, "b_max": 1.0},
]
val = torch.tensor([[[[50.0]]]])  # (1,1,1,1), midpoint → should map to 0.0
out_b = MultiScaleIntensityRange(windows=windows_custom_b)(val)
assert abs(out_b.item() - 0.0) < 0.01, f"Midpoint should map to 0.0, got {out_b.item()}"
print(f"[PASS] Custom b_min/b_max: midpoint→{out_b.item():.4f}")

# 5) Clip behaviour
out_clip = MultiScaleIntensityRange(windows=[{"a_min": 0.0, "a_max": 1.0}], clip=True)(torch.tensor([[[[2.0]]]]))
assert out_clip.item() == 1.0
out_noclip = MultiScaleIntensityRange(windows=[{"a_min": 0.0, "a_max": 1.0}], clip=False)(torch.tensor([[[[2.0]]]]))
assert out_noclip.item() == 2.0
print("[PASS] clip=True clips, clip=False does not")

# 6) Validation: providing both forms should fail
try:
    MultiScaleIntensityRange(windows=[{"a_min": 0, "a_max": 1, "window_center": 0, "window_width": 1}])
    assert False, "Should have raised"
except AssertionError as e:
    assert "exactly one" in str(e)
    print("[PASS] Both forms rejected")

# 7) Validation: providing neither form should fail
try:
    MultiScaleIntensityRange(windows=[{"b_min": 0, "b_max": 1}])
    assert False, "Should have raised"
except AssertionError as e:
    assert "exactly one" in str(e)
    print("[PASS] Neither form rejected")

# 8) 3-D input should fail (must be 4-D)
try:
    MultiScaleIntensityRange(windows=windows_range)(torch.randn(8, 8, 8))
    assert False
except AssertionError as e:
    assert "4-D" in str(e)
    print("[PASS] 3-D input rejected")

# 9) Multi-channel (C=2), concat mode → (W*C, Z Y X) = (6, 8, 8, 8)
img_2ch = torch.stack(
    [
        torch.linspace(-1024, 3071, 8**3).reshape(8, 8, 8),
        torch.linspace(3071, -1024, 8**3).reshape(8, 8, 8),
    ]
)  # (2, 8, 8, 8)
out_c = MultiScaleIntensityRange(windows=windows_range, mode="concat")(img_2ch)
assert out_c.shape == (6, 8, 8, 8), f"Expected (6,8,8,8), got {out_c.shape}"
# Concat order: [w0_ch0, w0_ch1, w1_ch0, w1_ch1, w2_ch0, w2_ch1]
print(f"[PASS] 2-ch concat: shape={tuple(out_c.shape)}")

# 10) Multi-channel (C=2), interleave mode → (C*W, Z Y X) = (6, 8, 8, 8)
out_i = MultiScaleIntensityRange(windows=windows_range, mode="interleave")(img_2ch)
assert out_i.shape == (6, 8, 8, 8), f"Expected (6,8,8,8), got {out_i.shape}"
# Interleave order: [ch0_w0, ch0_w1, ch0_w2, ch1_w0, ch1_w1, ch1_w2]
assert not torch.equal(out_c, out_i)
print(f"[PASS] 2-ch interleave: shape={tuple(out_i.shape)}, differs from concat")

# 11) Verify concat ordering: window-first grouping
#   concat[0] = w0_ch0, concat[1] = w0_ch1
#   Each window applied to full (C,D,H,W), so w0 output is (2,D,H,W) → first 2 channels
w0_only = MultiScaleIntensityRange(windows=[windows_range[0]])(img_2ch)  # (2, 8, 8, 8)
assert torch.equal(out_c[0], w0_only[0]), "concat[0] should be w0_ch0"
assert torch.equal(out_c[1], w0_only[1]), "concat[1] should be w0_ch1"
print("[PASS] Concat ordering verified (window-first)")

# 12) Verify interleave ordering: channel-first grouping
#   interleave[0] = ch0_w0, interleave[1] = ch0_w1, interleave[2] = ch0_w2
assert torch.equal(out_i[0], out_c[0]), "interleave[0]=ch0_w0 == concat[0]=w0_ch0"
assert torch.equal(out_i[1], out_c[2]), "interleave[1]=ch0_w1 == concat[2]=w1_ch0"
assert torch.equal(out_i[2], out_c[4]), "interleave[2]=ch0_w2 == concat[4]=w2_ch0"
assert torch.equal(out_i[3], out_c[1]), "interleave[3]=ch1_w0 == concat[1]=w0_ch1"
assert torch.equal(out_i[4], out_c[3]), "interleave[4]=ch1_w1 == concat[3]=w1_ch1"
assert torch.equal(out_i[5], out_c[5]), "interleave[5]=ch1_w2 == concat[5]=w2_ch1"
print("[PASS] Interleave ordering verified (channel-first)")

# 13) For C=1, concat == interleave
out_c1 = MultiScaleIntensityRange(windows=windows_range, mode="concat")(img_1ch)
out_i1 = MultiScaleIntensityRange(windows=windows_range, mode="interleave")(img_1ch)
assert torch.equal(out_c1, out_i1)
print("[PASS] C=1: concat == interleave")

# 14) Invalid mode rejected
try:
    MultiScaleIntensityRange(windows=windows_range, mode="bad")
    assert False
except AssertionError as e:
    assert "mode" in str(e)
    print("[PASS] Invalid mode rejected")

print("\nAll MultiScaleIntensityRange tests passed!\n")


# --- Test MultiScaleIntensityRanged (dict transform) ---

windows = [
    {"window_center": -600.0, "window_width": 1500.0},
    {"window_center": 50.0, "window_width": 400.0},
    {"window_center": 400.0, "window_width": 1800.0},
]

# 15) Single-channel, concat mode
out = MultiScaleIntensityRanged(keys="image", windows=windows)({"image": img_1ch})
assert out["image"].shape == (3, 8, 8, 8)
print(f"[PASS] Dict 1-ch concat: shape={tuple(out['image'].shape)}")

# 16) Multi-channel, concat mode
out_c = MultiScaleIntensityRanged(keys="image", windows=windows, mode="concat")({"image": img_2ch})
assert out_c["image"].shape == (6, 8, 8, 8)
print(f"[PASS] Dict 2-ch concat: shape={tuple(out_c['image'].shape)}")

# 17) Multi-channel, interleave mode
out_i = MultiScaleIntensityRanged(keys="image", windows=windows, mode="interleave")({"image": img_2ch})
assert out_i["image"].shape == (6, 8, 8, 8)
assert not torch.equal(out_c["image"], out_i["image"])
print(f"[PASS] Dict 2-ch interleave: shape={tuple(out_i['image'].shape)}")

# 18) Dict transform == base transform
out_base = MultiScaleIntensityRange(windows=windows, mode="concat")(img_2ch)
assert torch.equal(out_base, out_c["image"])
out_base_i = MultiScaleIntensityRange(windows=windows, mode="interleave")(img_2ch)
assert torch.equal(out_base_i, out_i["image"])
print("[PASS] Dict transform matches base transform for both modes")

# 19) 3-D input rejected in dict transform
try:
    MultiScaleIntensityRanged(keys="image", windows=windows)({"image": torch.randn(8, 8, 8)})
    assert False
except AssertionError as e:
    assert "4-D" in str(e)
    print("[PASS] 3-D input rejected in dict transform")

print("\nAll MultiScaleIntensityRanged tests passed!")

[PASS] 1-ch concat: shape=(3, 8, 8, 8), range=[0.000, 1.000]
[PASS] window_center/window_width matches a_min/a_max
[PASS] Mixed window styles accepted
[PASS] Custom b_min/b_max: midpoint→0.0000
[PASS] clip=True clips, clip=False does not
[PASS] Both forms rejected
[PASS] Neither form rejected
[PASS] 3-D input rejected
[PASS] 2-ch concat: shape=(6, 8, 8, 8)
[PASS] 2-ch interleave: shape=(6, 8, 8, 8), differs from concat
[PASS] Concat ordering verified (window-first)
[PASS] Interleave ordering verified (channel-first)
[PASS] C=1: concat == interleave
[PASS] Invalid mode rejected

All MultiScaleIntensityRange tests passed!

[PASS] Dict 1-ch concat: shape=(3, 8, 8, 8)
[PASS] Dict 2-ch concat: shape=(6, 8, 8, 8)
[PASS] Dict 2-ch interleave: shape=(6, 8, 8, 8)
[PASS] Dict transform matches base transform for both modes
[PASS] 3-D input rejected in dict transform

All MultiScaleIntensityRanged tests passed!


# nbdev

In [ ]:
!nbdev_export